# SwiftletCare — train thử v4 trên Colab T4

Bản v3 đã đóng băng được giữ nguyên. Bộ v4: 3106 train, 13 val đủ bốn lớp.
**Val rất nhỏ, nhãn do agent gán ban đầu, cần người thứ hai duyệt.** Đây là
lượt train thử để kiểm cải thiện dữ liệu, không phải kết quả chính thức.

1. Bật kernel Colab GPU T4.
2. Tải `ai-vision/data/colab/swiftletcare_train_val_v4_pilot.zip` lên thư mục
   `MyDrive/SwiftletCare/` trên Google Drive.
3. Chạy các ô bên dưới. Không cài lại torch. Không chạy notebook trên CPU máy.

Giữ ảnh chim nhỏ, tăng imgsz lên 1280 và tắt mosaic để tránh thu nhỏ thêm.
Điều này chưa bảo đảm sửa được nhận dạng sai. Không dùng test để chọn tham số.


In [ ]:
import sys, torch
print(sys.version)
assert torch.cuda.is_available(), 'Select a Colab GPU kernel before training'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
%pip install -q ultralytics==8.4.178 supervision==0.30.9


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, yaml
ZIP = Path('/content/drive/MyDrive/SwiftletCare/swiftletcare_train_val_v4_pilot.zip')
EXPECTED_SHA256 = '105cd8cbecadbbc416ced0ec5346453cc4a0f73ea3374905a8315d2e0299f55c'
assert ZIP.is_file(), f'Upload {ZIP.name} to MyDrive/SwiftletCare first'
with ZIP.open('rb') as f:
    actual_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()
assert actual_sha256 == EXPECTED_SHA256, 'Dataset ZIP hash mismatch'
DATASET = Path('/content/swiftletcare_v4_pilot')
DATASET.mkdir(exist_ok=False)
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    for name in z.namelist():
        assert (DATASET / name).resolve().is_relative_to(DATASET.resolve()), 'Unsafe ZIP path'
    z.extractall(DATASET)
meta = json.loads((DATASET / 'dataset.json').read_text())
print(meta)
assert meta['trainImages'] == 3106 and meta['valImages'] == 13
config = yaml.safe_load((DATASET / 'data.yaml').read_text())
assert config['names'] == ['owl', 'rat', 'snake', 'swiftlet']
config['path'] = str(DATASET)
(DATASET / 'data.yaml').write_text(yaml.safe_dump(config))
print('Pilot only: second human label review pending; validation is too small for reliable estimates.')


In [ ]:
from datetime import datetime, timezone
import ultralytics, supervision
from ultralytics import YOLO
SETTINGS = {'model': 'yolov8n.pt', 'epochs': 50, 'imgsz': 1280, 'batch': 4, 'mosaic': 0.0, 'close_mosaic': 0, 'seed': 42, 'patience': 15}
RUN_NAME = 'yolov8n_v4_pilot_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PROJECT = Path('/content/drive/MyDrive/SwiftletCare/runs')
PROJECT.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_NAME, SETTINGS)
model = YOLO(SETTINGS['model'])
training = model.train(data=str(DATASET / 'data.yaml'), project=str(PROJECT), name=RUN_NAME,
                       device=0, workers=2, plots=True, exist_ok=False,
                       **{k:v for k,v in SETTINGS.items() if k != 'model'})
RUN = Path(training.save_dir)
print('Outputs saved to:', RUN)


In [ ]:
import csv
BEST = RUN / 'weights/best.pt'
assert BEST.is_file()
with BEST.open('rb') as f:
    weights_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()
with (RUN / 'results.csv').open() as f:
    history = [{k.strip(): v for k,v in row.items()} for row in csv.DictReader(f)]
completed = max(int(float(row['epoch'])) for row in history)
result_meta = {
    'status': 'trial_v4_pilot_not_frozen', 'checkpoint': 'best.pt',
    'sha256': weights_sha256, 'epochsCompleted': completed,
    'configuredEpochs': SETTINGS['epochs'], 'datasetZipSha256': actual_sha256,
    'settings': SETTINGS, 'python': sys.version.split()[0], 'torch': torch.__version__,
    'ultralytics': ultralytics.__version__, 'supervision': supervision.__version__,
    'notes': ['Small pilot validation only; second human label review pending.',
              'No test evaluation or automatic replacement of frozen v3 weights.']
}
(RUN / 'weights-v4-pilot.json').write_text(json.dumps(result_meta, indent=2))
print(json.dumps(result_meta, indent=2))
print('Bring back best.pt, results.csv and weights-v4-pilot.json from', RUN)
